In [ ]:
import pandas as pd
df = pd.read_csv('/content/combined_statslogExcluded.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,target
0,40,1,2,140,289,0,0,172,0,0.0,1,0
1,49,0,3,160,180,0,0,156,0,1.0,2,1
2,37,1,2,130,283,0,1,98,0,0.0,1,0
3,48,0,4,138,214,0,0,108,1,1.5,2,1
4,54,1,3,150,195,0,0,122,0,0.0,1,0


In [ ]:
# prompt: perform agglomerative clustering on df with 3 clusters, add these clusters as column

from sklearn.cluster import AgglomerativeClustering

X = df.drop('target',axis=1)

# Perform agglomerative clustering with 3 clusters
clustering = AgglomerativeClustering(n_clusters=3)
df['cluster_region'] = clustering.fit_predict(X)

display(df.head())


,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,target,cluster_region
0,40,1,2,140,289,0,0,172,0,0.0,1,0,2
1,49,0,3,160,180,0,0,156,0,1.0,2,1,2
2,37,1,2,130,283,0,1,98,0,0.0,1,0,0
3,48,0,4,138,214,0,0,108,1,1.5,2,1,2
4,54,1,3,150,195,0,0,122,0,0.0,1,0,2


In [ ]:
# prompt: calculate silhouette score of the cluster

from sklearn.metrics import silhouette_score

# Calculate the silhouette score
silhouette_avg = silhouette_score(X, df['cluster_region'])

print(f"The average silhouette score is: {silhouette_avg}")


The average silhouette score is: 0.4505544329687876


In [ ]:
df['cluster_region'].value_counts()

,count
cluster_region,
2,490
0,256
1,172


In [ ]:
from sklearn.utils import shuffle
df = shuffle(df)

In [ ]:
from sklearn.model_selection import train_test_split
X = df.drop('target',axis=1)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=y, random_state=123, test_size=0.2)

In [ ]:
import xgboost as xgb

xgb_clf = xgb.XGBClassifier(
    objective="binary:logistic",  # or "multi:softprob" for multiclass
    eval_metric="logloss",        # can be "mlogloss" or "auc" based on task
    use_label_encoder=False,
    n_estimators = 153, max_depth = 3, learning_rate = 0.2017, subsample = 0.94,
    colsample_bytree = 0.9488, gamma = 3.5160, reg_alpha = 0.7811, reg_lambda = 0.6871
)

xgb_clf.fit(X_train, y_train)


/usr/local/lib/python3.11/dist-packages/xgboost/core.py:158: UserWarning: [08:01:04] WARNING: /workspace/src/learner.cc:740: 
Parameters: { "use_label_encoder" } are not used.

  warnings.warn(smsg, UserWarning)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9488, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, gamma=3.516, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=0.2017, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=3,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=153,
              n_jobs=None, num_parallel_tree=None, random_state=None, ...)

In [ ]:
!pip install fairlearn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 240.0/240.0 kB 3.7 MB/s eta 0:00:00


In [ ]:
# ✅ Install if not already installed
# !pip install fairlearn
from sklearn.metrics import accuracy_score, precision_score, recall_score

from fairlearn.metrics import (MetricFrame,
    demographic_parity_difference,
    demographic_parity_ratio,
    equalized_odds_difference,
    equalized_odds_ratio, selection_rate
)

import pandas as pd

# Simulated example:
y_true = y_test.copy()
y_pred = xgb_clf.predict(X_test)
sensitive_features = X_test['cluster_region']

# ➤ Create the fairness metrics frame
mf = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate,
        'precision': precision_score,
        'recall': recall_score
    },
    y_true=y_true,
    y_pred=y_pred,
    sensitive_features=sensitive_features
)

print("Group-wise metrics:")
print(mf.by_group)

# ➤ Fairness differences and ratios
dp_diff = demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive_features)
dp_ratio = demographic_parity_ratio(y_true, y_pred, sensitive_features=sensitive_features)

eo_diff = equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive_features)
eo_ratio = equalized_odds_ratio(y_true, y_pred, sensitive_features=sensitive_features)

print("\nDemographic parity difference:", dp_diff)
print("Demographic parity ratio:", dp_ratio)

print("\nEqualized odds difference:", eo_diff)
print("Equalized odds ratio:", eo_ratio)



Group-wise metrics:
                accuracy  selection_rate  precision    recall
cluster_region                                               
0               0.842105        0.526316   0.866667  0.838710
1               0.878788        0.939394   0.935484  0.935484
2               0.851064        0.425532   0.825000  0.825000

Demographic parity difference: 0.5138620245003225
Demographic parity ratio: 0.45298558682223744

Equalized odds difference: 0.8703703703703703
Equalized odds ratio: 0.12962962962962962


In [ ]:
reduced_df = df.drop(columns=['chol','restecg','slope', 'oldpeak'],axis=1)
X = reduced_df.drop('target',axis=1)
y = reduced_df['target']
X_train, X_test, y_train, y_test = train_test_split(X,y,stratify=y,test_size=0.2,random_state=123)


In [ ]:
from sklearn.ensemble import RandomForestClassifier
rfc_clf = RandomForestClassifier(n_estimators=110, max_depth=20, min_samples_split=4, min_samples_leaf=8,
                                 max_features='sqrt',bootstrap=True)
rfc_clf.fit(X_train, y_train)

RandomForestClassifier(max_depth=20, min_samples_leaf=8, min_samples_split=4,
                       n_estimators=110)

In [ ]:
# ✅ Install if not already installed
# !pip install fairlearn


# Simulated example:
y_true = y_test.copy()
y_pred = rfc_clf.predict(X_test)
sensitive_features = X_test['cluster_region']

# ➤ Create the fairness metrics frame
mf = MetricFrame(
    metrics={
        'accuracy': accuracy_score,
        'selection_rate': selection_rate,
        'precision': precision_score,
        'recall': recall_score
    },
    y_true=y_true,
    y_pred=y_pred,
    sensitive_features=sensitive_features
)

print("Group-wise metrics:")
print(mf.by_group)

# ➤ Fairness differences and ratios
dp_diff = demographic_parity_difference(y_true, y_pred, sensitive_features=sensitive_features)
dp_ratio = demographic_parity_ratio(y_true, y_pred, sensitive_features=sensitive_features)

eo_diff = equalized_odds_difference(y_true, y_pred, sensitive_features=sensitive_features)
eo_ratio = equalized_odds_ratio(y_true, y_pred, sensitive_features=sensitive_features)

print("\nDemographic parity difference:", dp_diff)
print("Demographic parity ratio:", dp_ratio)

print("\nEqualized odds difference:", eo_diff)
print("Equalized odds ratio:", eo_ratio)



Group-wise metrics:
                accuracy  selection_rate  precision    recall
cluster_region                                               
0               0.824561        0.508772   0.862069  0.806452
1               0.848485        0.909091   0.933333  0.903226
2               0.797872        0.436170   0.756098  0.775000

Demographic parity difference: 0.4729206963249516
Demographic parity ratio: 0.47978723404255325

Equalized odds difference: 0.8461538461538461
Equalized odds ratio: 0.15384615384615385


Next part: SHAP and DiCE on final models: XGBoost (Clinical) and RF (Non-clinical) https://colab.research.google.com/drive/18tMdCHef42eE_bSUJLB2anWoTwU6kWFZ?usp=sharing